In [3]:
# SupportGenie AI — Kaggle Notebook Template
# Customer Support Automation Agent (Enterprise Track)
# Demonstrates: Multi-agent system, custom tools, session & memory, observability, model training
# NOTE: Replace LLM_API_CALL() stub with actual Gemini/OpenAI calls (do not include API keys in public repo)

# =============== 0. Requirements & Setup ===============
# If running locally or needing extra packages, uncomment below:
# !pip install scikit-learn pandas joblib nltk sentence-transformers faiss-cpu

# =============== 1. Imports ===============
import os
import json
import logging
from typing import Dict, Any, List, Tuple
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
from collections import defaultdict
import uuid
import time

# optional: for better embeddings/retrieval if you choose
# from sentence_transformers import SentenceTransformer
# import faiss

# =============== 2. Logging / Observability ===============
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s | %(message)s",
)
logger = logging.getLogger("SupportGenie")

# Simple in-memory metrics collector
METRICS = defaultdict(int)

# =============== 3. Sample Dataset (or load your own) ===============
# Expected CSV format: message, category
# If you have a file use: df = pd.read_csv('/kaggle/input/.../support_tickets.csv')
def generate_sample_data(n=400):
    # small synthetic dataset for demo. Replace with real dataset for submission.
    msgs = [
        ("I can't login to my account. It says invalid password.", "account"),
        ("How do I request a refund for my last order?", "refund"),
        ("My payment failed but money was deducted.", "billing"),
        ("App crashes when I try to open it.", "technical"),
        ("Where is my order? It's delayed.", "order_status"),
        ("How do I change my shipping address?", "account"),
        ("I want to cancel my subscription.", "billing"),
    ]
    rows = []
    for i in range(n):
        msg, cat = msgs[i % len(msgs)]
        rows.append({"message": msg, "category": cat})
    return pd.DataFrame(rows)

try:
    # prefer user-provided dataset if available
    df = pd.read_csv("support_tickets.csv")
    logger.info("Loaded support_tickets.csv")
except Exception:
    logger.info("support_tickets.csv not found — generating sample dataset")
    df = generate_sample_data(600)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

# =============== 4. Train Intent Classifier ===============
# TF-IDF + Logistic Regression (simple, explainable)
TEXT_COL = "message"
LABEL_COL = "category"

X = df[TEXT_COL].astype(str)
y = df[LABEL_COL].astype(str)

vectorizer = TfidfVectorizer(stop_words="english", max_df=0.9, ngram_range=(1,2))
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
logger.info("Classification report:\n" + classification_report(y_test, preds))
METRICS['intent_accuracy'] = accuracy_score(y_test, preds)

# Save model & vectorizer for reuse
joblib.dump(clf, "support_intent_clf.pkl")
joblib.dump(vectorizer, "support_vectorizer.pkl")
logger.info("Saved intent classifier and vectorizer.")

# =============== 5. Build a Simple Knowledge Base (KB) ===============
# For a production project, ingest docs, FAQs, policies into a retriever (FAISS, ElasticSearch, Pinecone, etc.)
# Here we create a small KB (list of Q/A).
kb = [
    {"id":"kb_1", "title":"Reset password", "content":"To reset your password go to Settings > Reset Password, or click 'Forgot password' on login. We will send a reset link to your registered email."},
    {"id":"kb_2", "title":"Refund policy", "content":"Refunds are processed within 5-7 business days after approval. Please provide your order id and reason for refund."},
    {"id":"kb_3", "title":"Payment failed but charged", "content":"If payment failed but was charged, contact support with transaction ID. We will verify and refund if necessary."},
    {"id":"kb_4", "title":"App crashes", "content":"Please update the app to the latest version. If the problem persists, send logs and device model."},
    {"id":"kb_5", "title":"Order status", "content":"Use the 'Track Order' link in the orders page, or provide order id and we will check the status."},
]
kb_df = pd.DataFrame(kb)

# Simple TF-IDF retriever for KB: vectorize KB and find top match
kb_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
kb_vectors = kb_vectorizer.fit_transform(kb_df['content'].astype(str))

def retrieve_kb(query: str, top_k=2) -> List[Dict[str,Any]]:
    qv = kb_vectorizer.transform([query])
    sims = (kb_vectors @ qv.T).toarray().squeeze()
    idxs = np.argsort(-sims)[:top_k]
    results = []
    for i in idxs:
        results.append({
            "id": kb_df.loc[i,'id'],
            "title": kb_df.loc[i,'title'],
            "content": kb_df.loc[i,'content'],
            "score": float(sims[i])
        })
    METRICS['kb_queries'] += 1
    return results

# quick test
print(retrieve_kb("I forgot my password and can't login", top_k=2))

# =============== 6. LLM Wrapper (PLACEHOLDER) ===============
# Replace the contents of LLM_API_CALL with actual calls to Gemini or OpenAI.
# IMPORTANT: Do NOT put API keys in public notebooks. Use Kaggle Secrets or environment variables.

def LLM_API_CALL(prompt: str, model: str="gemini-proxy", max_tokens: int=256) -> str:
    """
    Placeholder wrapper.
    Replace this function body with your provider's SDK call:
    - For OpenAI: openai.ChatCompletion.create(...)
    - For Google Gemini via Vertex AI: use google.cloud.aiplatform and Vertex SDK or ADK
    The function must return a string (LLM's answer).
    """
    # SAFETY: No keys here. Instead we simulate a simple response for demo.
    logger.debug("LLM_API_CALL called with prompt length %d", len(prompt))
    METRICS['llm_calls'] += 1
    # Simple heuristic summarizer fallback (for demo only)
    if "summarize" in prompt.lower():
        return "Summary: user reports an account login issue; suggest reset password steps."
    if "generate response" in prompt.lower():
        return ("Hi! I'm sorry you're seeing this issue. "
                "Please try resetting your password by clicking 'Forgot password'. "
                "If that doesn't help, reply with your registered email and order id, and we'll escalate.")
    # fallback:
    return "LLM placeholder response. Replace LLM_API_CALL with a real LLM invocation."

# =============== 7. Session & Memory Manager ===============
class SessionMemory:
    """Simple in-memory session store and long-term patterns store"""
    def __init__(self):
        self.sessions = {}  # session_id -> list of messages
        self.long_term = {}  # user_id -> summary

    def create_session(self, user_id: str) -> str:
        sid = str(uuid.uuid4())
        self.sessions[sid] = {"user_id": user_id, "history": []}
        return sid

    def add_message(self, sid: str, role: str, message: str):
        self.sessions[sid]["history"].append({"role": role, "message": message, "ts": time.time()})

    def get_history(self, sid: str):
        return self.sessions[sid]["history"]

    def set_long_term(self, user_id: str, summary: str):
        self.long_term[user_id] = summary

    def get_long_term(self, user_id: str):
        return self.long_term.get(user_id)

memory = SessionMemory()

# =============== 8. Agent Implementations ===============
# We'll implement modular agents: InputAgent, ClassificationAgent, KBAgent, ResponseAgent, EscalationAgent

class InputAgent:
    """Parses/normalizes user input and pulls out quick metadata."""
    def __init__(self):
        pass

    def run(self, user_text: str) -> Dict[str,Any]:
        # Basic extraction heuristics (expandable with NER or LLM)
        out = {"text": user_text, "extracted": {}}
        if "order" in user_text.lower():
            out["extracted"]["has_order"] = True
        if "password" in user_text.lower() or "login" in user_text.lower():
            out["extracted"]["possible_issue"] = "account"
        # sentiment simple heuristic
        negative_words = ['not', "can't", "can't", 'angry', 'upset', 'frustrat']
        out['sentiment_negative'] = any(w in user_text.lower() for w in negative_words)
        METRICS['input_messages'] += 1
        logger.info("InputAgent extracted: %s", out["extracted"])
        return out

class ClassificationAgent:
    """Uses trained classifier to predict ticket category."""
    def __init__(self, clf, vectorizer):
        self.clf = clf
        self.vectorizer = vectorizer

    def run(self, text: str) -> Dict[str,Any]:
        x = self.vectorizer.transform([text])
        pred = self.clf.predict(x)[0]
        probs = None
        try:
            probs = self.clf.predict_proba(x)[0].tolist()
        except Exception:
            pass
        METRICS['classifications'] += 1
        logger.info("ClassificationAgent predicted: %s", pred)
        return {"category": pred, "probs": probs}

class KBAgent:
    """Retrieves best-matching KB entries using the retrieve_kb tool."""
    def __init__(self, retriever):
        self.retriever = retriever

    def run(self, text: str, top_k=2):
        res = self.retriever(text, top_k=top_k)
        logger.info("KBAgent found %d KB hits", len(res))
        METRICS['kb_hits'] += len(res)
        return res

class ResponseAgent:
    """Generates final user-facing reply using LLM + KB context."""
    def __init__(self, llm_fn):
        self.llm_fn = llm_fn

    def run(self, user_text: str, category: str, kb_results: List[Dict[str,Any]]) -> Dict[str,Any]:
        # Build prompt with context
        kb_text = "\n".join([f"- {r['title']}: {r['content']}" for r in kb_results])
        prompt = (
            f"You are an AI customer support assistant.\n"
            f"User message: {user_text}\n"
            f"Detected category: {category}\n"
            f"Knowledge base:\n{kb_text}\n\n"
            f"Task: generate a short, empathetic, step-by-step reply to help the user. "
            f"If further info is required, ask for it clearly. "
            f"Format:\nResponse:\n"
        )
        answer = self.llm_fn(prompt, model="gemini-proxy", max_tokens=256)
        logger.info("ResponseAgent produced an answer (len=%d)", len(answer))
        METRICS['responses'] += 1
        return {"response": answer}

class EscalationAgent:
    """Decides whether to escalate to human."""
    def __init__(self, escalation_threshold=0.5):
        self.escalation_threshold = escalation_threshold

    def run(self, category: str, sentiment_negative: bool, kb_results: List[Dict[str,Any]]) -> Dict[str,Any]:
        # Simple rules for escalation (replace with learned policy)
        should_escalate = False
        reason = None
        # escalate if negative sentiment + KB low-match score
        if sentiment_negative:
            if len(kb_results)==0 or kb_results[0]['score'] < 0.05:
                should_escalate = True
                reason = "negative sentiment + low KB confidence"
        # escalate if category is technical and KB not helpful
        if category == 'technical' and (len(kb_results)==0 or kb_results[0]['score'] < 0.05):
            should_escalate = True
            reason = "technical issue needs human"
        METRICS['escalations'] += int(should_escalate)
        logger.info("EscalationAgent decision: %s, reason: %s", should_escalate, reason)
        return {"escalate": should_escalate, "reason": reason}

# =============== 9. Orchestrator (Multi-Agent System) ===============
class AgentOrchestrator:
    def __init__(self, input_agent, classifier_agent, kb_agent, response_agent, escalation_agent, memory):
        self.input_agent = input_agent
        self.classifier_agent = classifier_agent
        self.kb_agent = kb_agent
        self.response_agent = response_agent
        self.escalation_agent = escalation_agent
        self.memory = memory

    def handle_message(self, user_id: str, user_text: str) -> Dict[str,Any]:
        # session management
        sid = None
        # create or reuse session for demo; in production link to user_id
        sid = self.memory.create_session(user_id)
        self.memory.add_message(sid, "user", user_text)

        # Run agents sequentially
        inp = self.input_agent.run(user_text)
        cls = self.classifier_agent.run(user_text)
        kb = self.kb_agent.run(user_text, top_k=3)
        esc = self.escalation_agent.run(cls['category'], inp['sentiment_negative'], kb)

        # If escalate -> produce front-loaded message and send details to human queue
        if esc['escalate']:
            human_msg = f"Escalation required. User: {user_id}. Category: {cls['category']}. Reason: {esc['reason']}. KB snippets: {[r['id'] for r in kb]}"
            logger.warning(human_msg)
            # For demo, LLM will still generate an initial reply acknowledging escalation.
            response_text = ("Thanks for the message — I've created a ticket and escalated this to our specialist team. "
                             "They will get back to you within 24 hours. Meanwhile, please provide order id/transaction id if available.")
            # store session & return
            self.memory.add_message(sid, "agent", response_text)
            return {"session_id": sid, "response": response_text, "escalated": True, "human_note": human_msg}

        # Otherwise generate response
        resp = self.response_agent.run(user_text, cls['category'], kb)
        final = resp['response']
        self.memory.add_message(sid, "agent", final)
        return {"session_id": sid, "response": final, "escalated": False, "category": cls['category'], "kb": kb}

# =============== 10. Instantiate Agents & Run Demo ===============
input_agent = InputAgent()
classification_agent = ClassificationAgent(clf, vectorizer)
kb_agent = KBAgent(retrieve_kb)
response_agent = ResponseAgent(LLM_API_CALL)
escalation_agent = EscalationAgent()
orchestrator = AgentOrchestrator(input_agent, classification_agent, kb_agent, response_agent, escalation_agent, memory)

# Demo messages
demo_messages = [
    ("user_1", "I can't log into my account. It says invalid password."),
    ("user_2", "I need a refund for order 12345 — it was the wrong size."),
    ("user_3", "The app keeps crashing when I upload a photo."),
]

for uid, text in demo_messages:
    out = orchestrator.handle_message(uid, text)
    print("USER:", text)
    print("AGENT RESPONSE:", out["response"])
    print("ESCALATED:", out["escalated"])
    print("-"*80)

# =============== 11. Export / Save Artifacts for Submission ===============
# Save KB, metrics and models (these files can be included in your GitHub/Kaggle Notebook output)
with open("kb.json", "w") as f:
    json.dump(kb, f, indent=2)
joblib.dump(kb_vectorizer, "kb_vectorizer.pkl")
joblib.dump(vectorizer, "client_vectorizer.pkl")
joblib.dump(clf, "intent_model.pkl")
with open("metrics.json", "w") as f:
    json.dump(METRICS, f, indent=2)

logger.info("Saved models, KB, and metrics. Notebook artifacts ready for upload.")

# =============== 12. README snippet (for your repo) ===============
README_SNIPPET = """
# SupportGenie AI — Customer Support Automation Agent

This repository contains the code for the SupportGenie AI capstone:
- Multi-agent system for customer support automation.
- Intent classification (TF-IDF + Logistic Regression).
- KB retrieval (TF-IDF retriever).
- LLM-based response generation (placeholder — plug your LLM).
- Session & memory manager.
- Observability via logging and metrics.

### How to run
1. Provide a `support_tickets.csv` with columns `message,category` or use the sample generator.
2. Replace `LLM_API_CALL()` implementation with your Gemini/OpenAI call (use Kaggle/Colab secrets).
3. Run all notebook cells. Save artifacts and push to GitHub or Kaggle Notebook.

### Features demonstrated
- Multi-agent orchestration
- Custom tool for KB retrieval
- Session & short-term memory
- Observability (logs, METRICS)
- Placeholder for LLM integration (Gemini recommended for bonus points)
"""
print(README_SNIPPET)


2025-11-30 07:15:57,509 INFO SupportGenie | support_tickets.csv not found — generating sample dataset
2025-11-30 07:15:57,551 INFO SupportGenie | Classification report:
              precision    recall  f1-score   support

     account       1.00      1.00      1.00        35
     billing       1.00      1.00      1.00        34
order_status       1.00      1.00      1.00        17
      refund       1.00      1.00      1.00        17
   technical       1.00      1.00      1.00        17

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120

2025-11-30 07:15:57,557 INFO SupportGenie | Saved intent classifier and vectorizer.
2025-11-30 07:15:57,570 INFO SupportGenie | InputAgent extracted: {'possible_issue': 'account'}
2025-11-30 07:15:57,572 INFO SupportGenie | ClassificationAgent predicted: account
2025-11-30 07:15:57,575 INFO SupportGenie | KBAgent found 3 KB hits
2025-11-30 

[{'id': 'kb_1', 'title': 'Reset password', 'content': "To reset your password go to Settings > Reset Password, or click 'Forgot password' on login. We will send a reset link to your registered email.", 'score': 0.49311775838793614}, {'id': 'kb_2', 'title': 'Refund policy', 'content': 'Refunds are processed within 5-7 business days after approval. Please provide your order id and reason for refund.', 'score': 0.0}]
USER: I can't log into my account. It says invalid password.
AGENT RESPONSE: LLM placeholder response. Replace LLM_API_CALL with a real LLM invocation.
ESCALATED: False
--------------------------------------------------------------------------------
USER: I need a refund for order 12345 — it was the wrong size.
AGENT RESPONSE: LLM placeholder response. Replace LLM_API_CALL with a real LLM invocation.
ESCALATED: False
--------------------------------------------------------------------------------
USER: The app keeps crashing when I upload a photo.
AGENT RESPONSE: LLM placehol

In [1]:
!pip install google-generativeai scikit-learn pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 8.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14